# SpaceX Falcon 9 – Exploratory Data Analysis with SQL
Load the SpaceX launch table into a SQLite database and answer the assignment questions with SQL queries.

In [1]:
import sqlite3
import pandas as pd

In [2]:
import os
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_2/data/Spacex.csv"
df = pd.read_csv("data/Spacex.csv" if os.path.exists("data/Spacex.csv") else URL)
con = sqlite3.connect("my_data1.db")
df.to_sql("SPACEXTBL", con, if_exists="replace", index=False, method="multi")
# remove blank rows
con.execute("DROP TABLE IF EXISTS SPACEXTABLE")
con.execute("CREATE TABLE SPACEXTABLE AS SELECT * FROM SPACEXTBL WHERE Date IS NOT NULL")
print(pd.read_sql("SELECT COUNT(*) AS rows FROM SPACEXTABLE", con))

   rows
0   101


In [3]:
def sql(query):
    return pd.read_sql(query, con)

## Task 1: Names of the unique launch sites

In [4]:
sql("""SELECT DISTINCT Launch_Site FROM SPACEXTABLE""")

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


## Task 2: 5 records where launch sites begin with 'CCA'

In [5]:
sql("""SELECT * FROM SPACEXTABLE WHERE Launch_Site LIKE 'CCA%' LIMIT 5""")

,Date,Time,Booster_Version,Launch_Site,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,677,LEO (ISS),NASA (CRS),Success,No attempt


## Task 3: Total payload mass carried by boosters launched by NASA (CRS)

In [6]:
sql("""SELECT SUM(PAYLOAD_MASS__KG_) AS Total_Payload_kg FROM SPACEXTABLE WHERE Customer = 'NASA (CRS)'""")

,Total_Payload_kg
0,45596


## Task 4: Average payload mass carried by booster version F9 v1.1

In [7]:
sql("""SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_kg FROM SPACEXTABLE WHERE Booster_Version = 'F9 v1.1'""")

,Avg_Payload_kg
0,2928.4


## Task 5: Date of the first successful landing outcome on a ground pad

In [8]:
sql("""SELECT MIN(Date) AS First_Success_Ground_Pad FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (ground pad)'""")

,First_Success_Ground_Pad
0,2015-12-22


## Task 6: Boosters with success on drone ship and payload mass between 4000 and 6000

In [9]:
sql("""SELECT Booster_Version FROM SPACEXTABLE WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000""")

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


## Task 7: Total number of successful and failure mission outcomes

In [10]:
sql("""SELECT Mission_Outcome, COUNT(*) AS Total FROM SPACEXTABLE GROUP BY Mission_Outcome""")

,Mission_Outcome,Total
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


## Task 8: Booster versions which have carried the maximum payload mass

In [11]:
sql("""SELECT DISTINCT Booster_Version FROM SPACEXTABLE WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTABLE)""")

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


## Task 9: Failure landing outcomes on drone ship in 2015, with month, booster version and launch site

In [12]:
sql("""SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTABLE WHERE substr(Date, 1, 4) = '2015' AND Landing_Outcome = 'Failure (drone ship)'""")

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


## Task 10: Rank the count of landing outcomes between 2010-06-04 and 2017-03-20 in descending order

In [13]:
sql("""SELECT Landing_Outcome, COUNT(*) AS Count FROM SPACEXTABLE WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY Count DESC""")

,Landing_Outcome,Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
